# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook provides a step-by-step walkthrough for loading, exploring, and processing the **FAIR^2 dataset** using the [`mlcroissant`](https://github.com/mlcommons/croissant) library, explicitly referring to all fields, record sets, and columns by their `@id` fields as per Croissant best practices. The dataset is distributed via a Croissant schema hosted at the provided URL.

### Dataset Source
**Croissant schema URL:**  
[https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json](https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json)


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -q mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Define the dataset URL (Croissant schema)
url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the Dataset object
dataset = mlc.Dataset(url)

# Print summary metadata
md = dataset.metadata
print(f"Dataset Title: {md.name}\n")
print(f"Identifier: {md.identifier}")
print(f"Description: {md.description}\n")
print(f"Dataset version: {md.version}")
print(f"License: {md.license}\n")

## 2. Data Overview
Review available Record Sets and Fields. All references are by `@id`.

Let's enumerate the record sets, and for each record set, show their fields and associated column `@id`s.

In [ ]:
# Get all record set @ids
record_sets = dataset.record_sets

print(f"Number of record sets: {len(record_sets)}\n")
for rs in record_sets:
    # Print the record set's @id and name
    print(f"Record Set: {rs['@id']}")
    if 'name' in rs:
        print(f"  Name: {rs['name']}")
    if 'description' in rs:
        print(f"  Description: {rs['description']}")
    # Print all fields with their ids, names, and associated columns
    if 'field' in rs and isinstance(rs['field'], list):
        print(f"  Fields:")
        for field in rs['field']:
            # The field object may be directly dict or via entity lookup, so resolve if needed
            field_obj = field if isinstance(field, dict) else dataset._lookup_by_id(field)
            print(f"    - Field @id: {field_obj['@id']}")
            if 'name' in field_obj:
                print(f"      name: {field_obj['name']}")
            if 'column' in field_obj:
                col = field_obj['column']
                # If column is an @id, look up for details
                if isinstance(col, str):
                    col_obj = dataset._lookup_by_id(col)
                    print(f"      column @id: {col_obj['@id']}")
                    if 'name' in col_obj:
                        print(f"      column name: {col_obj['name']}")
                elif isinstance(col, dict) and '@id' in col:
                    print(f"      column @id: {col['@id']}")
    print('-' * 50)


## 3. Data Extraction

We'll load data from all available record sets into Pandas DataFrames. Each key of the returned `dataframes` dictionary is the record set's `@id`.

Use the `@id` of the record sets as shown above.

In [ ]:
# Gather record set @ids
record_set_ids = [rs['@id'] for rs in record_sets]

dataframes = {}

# Load records for each record set
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded {len(df)} records from record set: {record_set_id}")
    print(f"Columns: {df.columns.tolist()}")
    print('-' * 30)

# Example: Show the first few records from the main clinical data record set
if len(record_set_ids) > 0:
    main_rs_id = record_set_ids[0]
    print(f"\nPreview of {main_rs_id}:")
    display(dataframes[main_rs_id].head())

## 4. Exploratory Data Analysis (EDA)

Let's perform some simple EDA by:
- Selecting a numeric field by its `@id` (e.g., interval between diagnosis, age, or similar).
- Filtering records, normalizing the numeric field, and grouping by another field (if available).

Choose field and group `@id`s as shown above for reproducibility.

In [ ]:
# Set the record set @id and the numeric field @id for EDA
# Please replace these with actual @id values from your Data Overview output
record_set_id = record_set_ids[0]  # Use the first record set for demo
df = dataframes[record_set_id]

# Let's try to automatically infer a suitable numeric field @id
numeric_field_candidates = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
if not numeric_field_candidates:
    # If all columns are object/string, try converting columns with likely numeric content
    for col in df.columns:
        try:
            df[col] = pd.to_numeric(df[col])
        except Exception:
            continue
    numeric_field_candidates = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]

if numeric_field_candidates:
    numeric_field_id = numeric_field_candidates[0]  # Select the first numeric column (by @id)
    print(f"Using numeric field: {numeric_field_id}")
else:
    print("No numeric field available for EDA in this record set.")

# Filter and normalize as in the template
if numeric_field_candidates:
    threshold = df[numeric_field_id].mean()  # Use mean as a threshold for demonstration
    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
    display(filtered_df.head())

    # Normalized field
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - \
        filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"\nNormalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Group by a categorical/grouping field, using @id, if any string column
    group_field_candidates = [col for col in df.columns if pd.api.types.is_object_dtype(df[col])]
    if group_field_candidates:
        group_field_id = group_field_candidates[0]
        print(f"\nGrouping by: {group_field_id}")
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
        print(grouped_df.head())
else:
    print('No numeric data to analyze in this record set.')

## 5. Visualization

Visualize the distribution of the selected numeric field and its relationship with a chosen group field. This will use the `@id` fields.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style="whitegrid")

if numeric_field_candidates:
    plt.figure(figsize=(6,4))
    sns.histplot(df[numeric_field_id], bins=10, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Frequency')
    plt.show()

    # Show a boxplot by group field if appropriate
    if group_field_candidates:
        plt.figure(figsize=(8,4))
        sns.boxplot(x=df[group_field_id], y=df[numeric_field_id])
        plt.xticks(rotation=45)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.tight_layout()
        plt.show()

## 6. Conclusion

- The **FAIR^2 dataset** contains detailed clinical, molecular, and pathological data from cancer survivors with second primary colorectal cancer.
- We loaded and analyzed the data fully by referencing record sets and fields via their `@id`s using the `mlcroissant` library.
- Basic exploratory analysis and visualizations were demonstrated, establishing a reproducible template for further, domain-specific analysis and modeling.

#### Next steps:
- Further clinical/biostatistical analysis of MSI-H status or anatomical distribution using field `@id` references.
- Integration with downstream ML pipelines by programmatic referencing of data structures by Croissant metadata.
- Custom EDA or hypothesis testing driven by the research questions, leveraging the rich, standardized metadata.
